In [8]:
# v_uw_kpis_yearly_nep_millions.py
# ONE CSV, yearly grain, with Net Earned Premium in *millions* USD (integer, no decimals).
# Columns: Company, LOB, Date, Currency, State, NetEarnedPremium_MUSD

import pandas as pd
import numpy as np
from itertools import product
from datetime import date

np.random.seed(42)

# -----------------------------
# Entities
# -----------------------------
COMPANIES = [
    "Atlas Mutual Insurance",
    "Seaboard Indemnity Co.",
    "HarborPoint Casualty",
    "Pioneer Assurance Group",
    "Vista National Insurance",
    "Pacific Crest Underwriters",
    "Mariner General",
    "Cedar Ridge Insurance",
    "Blue Harbor Assurance",
    "Heritage Specialty",
    "Northeast Fidelity",
    "Coastal Shield Insurance",
    "Summit Peak Casualty",
    "Liberty Bay Indemnity",
    "Crescent National",
]

LOBS = ["Personal Auto", "Homeowners", "Workers Comp", "Commercial Auto"]

# 10 coastal states (east + west)
STATES = ["CA", "OR", "WA", "NY", "NJ", "MA", "MD", "VA", "NC", "FL"]

CURRENCY = "USD"
YEARS = [2023, 2024]

# -----------------------------
# Industry targets (BILLIONS USD)
# We'll convert to *millions* and allocate integers.
# -----------------------------
TARGETS_B = {
    2023: {
        "Personal Auto": (285, 315),
        "Homeowners": (160, 173),
        "Workers Comp": (44, 44),
        "Commercial Auto": (62, 62),
    },
    2024: {
        "Personal Auto": (300, 330),
        "Homeowners": (173, 173),
        "Workers Comp": (46.3, 46.3),
        "Commercial Auto": (64, 67),
    },
}

def sample_total_millions(year, lob):
    lo_b, hi_b = TARGETS_B[year][lob]
    if abs(hi_b - lo_b) < 1e-12:
        total_b = lo_b
    else:
        total_b = np.random.uniform(lo_b, hi_b)
    return int(round(total_b * 1000))  # billions → millions, rounded to integer MUSD

def allocate_integer_shares(total_units, weights):
    """
    Allocate 'total_units' integer units according to 'weights' (sum ~1),
    by largest remainder method to ensure the integer allocations sum exactly.
    """
    weights = np.array(weights, dtype=float)
    weights /= weights.sum()
    raw = weights * total_units
    floor = np.floor(raw).astype(int)
    remainder = total_units - floor.sum()
    # Distribute remaining ones to the largest fractional parts
    frac = raw - floor
    order = np.argsort(-frac)  # descending
    floor[order[:remainder]] += 1
    return floor

rows = []

for yr in YEARS:
    dt = date(yr, 12, 31)

    for lob in LOBS:
        # National target in *millions* USD (integer)
        national_nep_musd = sample_total_millions(yr, lob)

        # Company weights for this LOB/year (mild skew for larger carriers)
        alpha_comp = 1.0 + np.random.uniform(0.2, 1.2, size=len(COMPANIES))
        comp_w = np.random.dirichlet(alpha_comp)
        comp_musd = allocate_integer_shares(national_nep_musd, comp_w)

        # Within each company, allocate across states (slight skew for CA/FL/NY)
        state_alpha = np.array([1.6 if s in ["CA", "FL", "NY"] else 1.0 for s in STATES])

        for comp_idx, company in enumerate(COMPANIES):
            if comp_musd[comp_idx] == 0:
                # Emit zero rows (keeps the full grain intact)
                st_alloc = np.zeros(len(STATES), dtype=int)
            else:
                st_w = np.random.dirichlet(state_alpha)
                st_alloc = allocate_integer_shares(comp_musd[comp_idx], st_w)

            for state, musd in zip(STATES, st_alloc):
                rows.append({
                    "Company": company,
                    "LOB": lob,
                    "Date": dt,               # year-end date
                    "Currency": CURRENCY,
                    "State": state,
                    "NetEarnedPremium_MUSD": int(musd)  # integer millions, no decimals
                })

df = pd.DataFrame(rows).sort_values(
    ["Company", "LOB", "State", "Date"]
).reset_index(drop=True)

out_csv = "v_uw_kpis_yearly_nep_millions.csv"
df.to_csv(out_csv, index=False)
print(f"Wrote {len(df):,} rows to {out_csv}")

# --- Optional sanity check (comment in if you want) ---
# for yr in YEARS:
#     for lob in LOBS:
#         mask = (df["Date"].dt.year == yr) & (df["LOB"] == lob)
#         total_m = df.loc[mask, "NetEarnedPremium_MUSD"].sum()
#         print(yr, lob, f"{total_m:,} MUSD (≈ ${total_m/1000:.1f}B)")


Wrote 1,200 rows to v_uw_kpis_yearly_nep_millions.csv


## Net Writter Premium